**Installing OpenAI Package**
To use the OpenAI API, we need to install the `openai` package. We can do this using the `pip` command.

In [ ]:
import subprocess
import sys

def install_openai():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openai"])

install_openai()

**OpenAI API & Functions**
Pizza ordeing chat demo.
Ex. Add another function and execue it during the chat.

In [5]:
import json
from openai import OpenAI

key = "OPEN-AI-KEY"

client = OpenAI(api_key=key)


def place_order(name, size, take_away=False):
    place = "take away" if take_away else "eat in"
    print("\033[31mplace_order function is executed:\033[0m", end=" ")
    print(f"Placing order for {name} pizza, {size}, to {place}")


def call_staff():
    print("\033[31mcall_staff function is executed:\033[0m", end=" ")
    print("Calling staff to table")


def get_chat(messages, model="gpt-4.1-nano", temperature=0.2, tools=None):

    response = client.responses.create(
        model=model,
        input=messages,
        temperature=temperature,
        tools=tools,
    )

    output = response.output[0]

    # Check if the model wants to call a tool
    if output.type == "function_call":

        tool_name = output.name
        args = json.loads(output.arguments)

        f = globals()[tool_name]
        f(**args)

        return True

    else:
        print("\033[94mPizza helper:\033[0m", response.output_text)
        return False


def main():

    tools = [
    {
        "type": "function",
        "name": "place_order",
        "description": "Place an order for a pizza",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": "The name of the pizza"
                },
                "size": {
                    "type": "string",
                    "enum": ["small", "medium", "large"],
                    "description": "The size of the pizza. Always ask for clarification if not specified."
                },
                "take_away": {
                    "type": "boolean",
                    "description": "Whether the pizza is taken away. Assume false if not specified"
                }
            },
            "required": ["name", "size", "take_away"]
        }
    },
    {
        "type": "function",
        "name": "call_staff",
        "description": "Call a member of staff to the table",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    }
    ]

    messages = [
        {
            "role": "developer",
            "content": "Don't make assumptions about what values to put into functions. Ask for clarification if needed."
        }
    ]

    function_called = False

    print("\033[94mPizza helper:\033[0m", "How may I help you? Would you like a pizza?")

    try:
        while not function_called:

            user_input = input("You: ")

            messages.append({
                "role": "user",
                "content": user_input
            })

            function_called = get_chat(messages, tools=tools)

    except KeyboardInterrupt:
        print("\n\033[94mPizza helper:\033[0m", "Thank you. Bye Bye")


if __name__ == "__main__":
    main()

Pizza helper: How may I help you? Would you like a pizza?
Pizza helper: Would you like to specify the type of pizza and whether you'd like it for takeout?
place_order function is executed: Placing order for large pizza pizza, large, to take away
